### VPC Flow Log pre-processor: part of the DUNE project
Ingests VPC flow logs, labels known cloud provider IPs, tor IPs, and anything known you want to define in known_ips.csv. Flows are then aggregated into records with summed time spans, byte and packet counts. Flows are tagged north-south or east-west (tag-direction) and ingress or egress (flow-direction). Part of the DUNE project (https://github.com/opendr-io/dune) and useful for hunting threats that are resistant to conventional detection or just hunting at great scale.
   
Field definitions:

- tag-direction: north-south (involving external networks) or east-west (staying within private networks).

- flow-direction: ingress or egress, relative to the ENI, from the v5 flow log field. Older (v2/v3) logs don't carry it and it isn't derived, so the column and the direction filters are absent for them.

- action: accept or reject, according to the flow records

- tag-org: the organization the flow's public end belongs to -- cloud provider / service (AWS, Azure, Google, Akamai, Cloudflare, Fastly, GitHub, Oracle, Zscaler, Tor) or your own known_ips.csv name

- tag-cloud: which reference list the public end matched -- aws, zscaler, tor, azure, google, akamai, cloudflare, fastly, github or oracle (unchanged by known_ips.csv). East-west sessions get "private", or "reserved" (with tag-org left empty) when either end is in reserved space that isn't RFC1918 (100.64/10 CGNAT, documentation / benchmark ranges, ...)

- tag-service: service name from conf/tcp-ports.md / udp-ports.md for the session's listed port; "anomalous" when src and dst ports name two different services (scan traffic from a fixed well-known port)

- tag-src-public	srcaddr is a public IPv4 address
- tag-dst-public	dstaddr is a public IPv4 address

- tag-label: behavior tags, e.g. "beaconing" (cell 19)

Fields the notebook derives are prefixed `tag-` so they never collide with flow log fields. The flow log's own fields, such as `type` (IPv4 / IPv6 / EFA), are kept as-is.

In [ ]:
# --- import ---
import pandas as pd
import ipaddress
from ipaddress import ip_address
import numpy as np
import os
import json
import re

In [ ]:
PARQUET_PATH = "vflows-4m.parquet"

In [ ]:
# --- cell 00 ---
# load flow data
# PARQUET_PATH is either one .parquet file or a folder -- a folder loads every
# *.parquet directly inside it (not subfolders) and stacks them. Files are read one
# at a time so exports with different column types still combine; an unreadable
# file in a folder is skipped with a warning.
# start/end must be epoch seconds (Int64) -- cell 08's MERGE_GAP and the duration /
# beacon cells do plain-second arithmetic. Some exports store them as timestamps
# or durations instead; normalise those here so every cell sees the same type.
# An export may carry start/end OR start_time/end_time; the missing pair is derived
# from the other, so vflows always has all four.

def _norm_times(df):
    """start/end -> epoch seconds (Int64); start_time/end_time -> datetime64[ms, UTC].
    A file may carry only one of the pairs -- the other is derived from it."""
    for _a, _t in (("start", "start_time"), ("end", "end_time")):
        if _a in df.columns:
            _s = df[_a]
            if pd.api.types.is_datetime64_any_dtype(_s):
                _s = _s.dt.tz_localize("UTC") if _s.dt.tz is None else _s
                df[_a] = ((_s - pd.Timestamp(0, tz="UTC")) // pd.Timedelta(seconds=1)).astype("Int64")
            elif pd.api.types.is_timedelta64_dtype(_s):
                df[_a] = (_s // pd.Timedelta(seconds=1)).astype("Int64")
            else:
                df[_a] = pd.to_numeric(_s, errors="coerce").astype("Int64")
        if _t in df.columns:
            df[_t] = pd.to_datetime(df[_t], utc=True, errors="coerce").astype("datetime64[ms, UTC]")
        if _a in df.columns and _t not in df.columns:
            df[_t] = pd.to_datetime(df[_a].astype("float64"), unit="s", utc=True).astype("datetime64[ms, UTC]")
        elif _t in df.columns and _a not in df.columns:
            df[_a] = ((df[_t] - pd.Timestamp(0, tz="UTC")) // pd.Timedelta(seconds=1)).astype("Int64")
    return df

if os.path.isdir(PARQUET_PATH):
    _files = sorted(os.path.join(PARQUET_PATH, f) for f in os.listdir(PARQUET_PATH)
                    if f.lower().endswith(".parquet"))
    if not _files:
        raise FileNotFoundError(f"no .parquet files in folder {PARQUET_PATH!r}")
elif os.path.isfile(PARQUET_PATH):
    _files = [PARQUET_PATH]
else:
    raise FileNotFoundError(f"PARQUET_PATH {PARQUET_PATH!r} is not a file or folder")

_frames = []
for _f in _files:
    try:
        _frames.append(_norm_times(pd.read_parquet(_f)))
    except Exception as _e:
        if len(_files) == 1:
            raise
        print(f"!! skipped {os.path.basename(_f)} ({type(_e).__name__}: {_e})")
if not _frames:
    raise ValueError(f"none of the {len(_files)} parquet files in {PARQUET_PATH!r} could be read")

_nRead = len(_frames)
vflows = pd.concat(_frames, ignore_index=True) if _nRead > 1 else _frames[0]
del _frames

# srcaddr / dstaddr -> src-int / dst-int (UInt32): the IPv4 address as an integer, next
# to the string column (kept for reading). Range checks and CIDR matching downstream use
# these, so nothing after this cell parses address strings. One strict IPv4 match per
# distinct address, mapped back to every row; IPv6, missing and malformed -> <NA>.

_OCTET = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
_IPV4  = rf"{_OCTET}(?:\.{_OCTET}){{3}}"

def ipv4_to_uint32(s):
    """Series of address strings -> UInt32 Series (same index); non-IPv4 -> <NA>."""
    codes, uniq = pd.factorize(s.astype("string"))
    u    = pd.Series(uniq, dtype="string")
    ok   = u.str.fullmatch(_IPV4).fillna(False).to_numpy(bool)
    vals = np.zeros(len(u), dtype="uint64")
    if ok.any():
        o = u[ok].str.split(".", expand=True).astype("uint64").to_numpy()
        vals[ok] = (o[:, 0] << np.uint64(24) | o[:, 1] << np.uint64(16)
                    | o[:, 2] << np.uint64(8) | o[:, 3])
    arr = pd.array(vals.astype("uint32"), dtype="UInt32")
    arr[~ok] = pd.NA
    return pd.Series(arr.take(codes, allow_fill=True), index=s.index)   # code -1 (missing) -> <NA>

_missing = [c for c in ("srcaddr", "dstaddr") if c not in vflows.columns]
if _missing:
    raise KeyError(f"not in vflows: {_missing}")
for _a, _i in (("srcaddr", "src-int"), ("dstaddr", "dst-int")):
    vflows.insert(vflows.columns.get_loc(_a) + 1, _i, ipv4_to_uint32(vflows[_a]))

# The pipeline is IPv4-only (reference CIDR lists too), so IPv6 flows would get no
# tag-direction, public flag or cloud label and drop out of north-south views unseen.
# Say so loudly if any turn up; "-" (NODATA / SKIPDATA) isn't counted as malformed.
for _a, _i in (("srcaddr", "src-int"), ("dstaddr", "dst-int")):
    _bad = vflows.loc[vflows[_i].isna() & vflows[_a].notna(), _a].astype("string").str.strip()
    _bad = _bad[_bad.ne("-") & _bad.ne("")]
    _v6  = _bad.str.contains(":", regex=False)
    if _v6.any():
        print(f"!! {int(_v6.sum()):,} IPv6 addresses in {_a} -- NOT handled: those flows get no "
              "tag-direction, public flag or cloud label")
    if (~_v6).any():
        print(f"!! {int((~_v6).sum()):,} malformed {_a} values -> <NA>, e.g. "
              + ", ".join(repr(x) for x in _bad[~_v6].unique()[:3]))

print(f"loaded {len(vflows):,} rows from {PARQUET_PATH!r}"
      + (f" ({_nRead} of {len(_files)} files)" if os.path.isdir(PARQUET_PATH) else ""))
_tc = [c for c in ("start", "end", "start_time", "end_time") if c in vflows.columns]
print(vflows[_tc].dtypes.to_string())
print(f"IPv4 parsed: src {int(vflows['src-int'].notna().sum()):,}, "
      f"dst {int(vflows['dst-int'].notna().sum()):,} of {len(vflows):,} rows "
      "(the rest are IPv6 / missing -> <NA>)")
print(f"shape: {vflows.shape}")

In [ ]:
# --- cell 01 ---
# All reference CIDR data lives here (local snapshots, no network needed)

CIDR_DIR   = "cidrs"
_REF_COLS  = ["ip_prefix", "region", "service", "network_border_group"]

# Load AWS IP ranges from the local snapshot in cidrs/.
# Missing / unreadable file -> warn and carry on with an empty frame.

try:
    with open(os.path.join(CIDR_DIR, "ip-ranges.json")) as _f:
        aws_data = json.load(_f)
    aws = pd.DataFrame(aws_data["prefixes"])[_REF_COLS]
except (OSError, ValueError, KeyError) as _e:
    print(f"!! AWS ip-ranges.json not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    aws = pd.DataFrame(columns=_REF_COLS)

print(f"{len(aws)} AWS CIDRs loaded")

In [ ]:
# --- cell 02 ---
# Load the already-normalized Zscaler CIDR list from cidrs/zscaler.csv.
# Schema matches `aws`: ip_prefix, region, service, network_border_group.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
try:
    zscaler = pd.read_csv(os.path.join(CIDR_DIR, "zscaler.csv"))
    zscaler = zscaler[_SCHEMA]
except (OSError, ValueError, KeyError) as _e:
    print(f"!! zscaler.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    zscaler = pd.DataFrame(columns=_SCHEMA)

print(f"{len(zscaler)} Zscaler CIDRs loaded")

In [ ]:
# --- cell 03 ---
# Load the already-normalized Tor exit list from cidrs/tor.csv.
# Schema matches `aws`; each entry is a /32 host. service = "TOR".
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
try:
    tor = pd.read_csv(os.path.join(CIDR_DIR, "tor.csv"))
    tor = tor[_SCHEMA]
except (OSError, ValueError, KeyError) as _e:
    print(f"!! tor.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    tor = pd.DataFrame(columns=_SCHEMA)

print(f"{len(tor)} Tor exit IPs loaded")

In [ ]:
# --- cell 04 ---
# Azure Service Tags (cidrs/azure.json) -> same schema as `aws`.
# Structure differs from AWS: values[] = service tags, CIDRs nested in
# properties.addressPrefixes[] -> explode. service = systemService, falling back
# to the tag `name` when systemService is blank. IPv6 dropped. One row per CIDR:
# a specific tag beats the AzureCloud umbrella, and a tag with a region beats one without.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
try:
    with open(os.path.join(CIDR_DIR, "azure.json")) as _f:
        _az = json.load(_f)

    _rows = []
    for _v in _az["values"]:
        _p   = _v["properties"]
        _svc = _p.get("systemService") or _v["name"]
        _reg = _p.get("region") or ""
        for _pfx in _p.get("addressPrefixes", []):
            if ":" in _pfx:                       # IPv4 only
                continue
            _rows.append({"ip_prefix": _pfx, "region": _reg, "service": _svc,
                          "_umbrella": _v["name"].startswith("AzureCloud")})

    azure = (pd.DataFrame(_rows)
             .sort_values(["ip_prefix", "_umbrella", "region"],
                          ascending=[True, True, False], kind="stable")
             .drop_duplicates("ip_prefix", keep="first")
             .drop(columns="_umbrella")
             .reset_index(drop=True))
    azure["network_border_group"] = pd.NA
    azure["region"]  = azure["region"].replace("", pd.NA)
    azure = azure[_SCHEMA]
    _chg = _az.get("changeNumber")
except (OSError, ValueError, KeyError) as _e:
    print(f"!! azure.json not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    azure, _chg = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(azure)} Azure CIDRs loaded ({azure['service'].nunique()} services, changeNumber {_chg})")

In [ ]:
# --- cell 05 ---
# Google ranges: goog.json (all Google) + gcloud.json (GCP customer ranges, a
# subset of goog) -> same schema as `aws`. IPv4 only. service/region come from
# gcloud; the goog-only remainder is a generic "GOOGLE". A specific gcloud prefix
# beats a broad goog block via most-specific-match downstream; an exact-duplicate
# prefix keeps the gcloud row (it carries service + region).
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
try:
    with open(os.path.join(CIDR_DIR, "gcloud.json")) as _f:
        _gc = json.load(_f)
    with open(os.path.join(CIDR_DIR, "goog.json")) as _f:
        _gg = json.load(_f)

    _rows = [{"ip_prefix": _p["ipv4Prefix"], "region": _p.get("scope"),
              "service": _p.get("service") or "Google Cloud", "_src": "gcloud"}
             for _p in _gc["prefixes"] if "ipv4Prefix" in _p]
    _rows += [{"ip_prefix": _p["ipv4Prefix"], "region": pd.NA,
               "service": "GOOGLE", "_src": "goog"}
              for _p in _gg["prefixes"] if "ipv4Prefix" in _p]

    google = (pd.DataFrame(_rows)
              .sort_values("_src", kind="stable")          # gcloud rows before goog
              .drop_duplicates("ip_prefix", keep="first")
              .drop(columns="_src")
              .reset_index(drop=True))
    google["network_border_group"] = pd.NA
    google["region"]  = google["region"].replace("", pd.NA)
    google = google[_SCHEMA]
    _sync = _gc.get("syncToken")
except (OSError, ValueError, KeyError) as _e:
    print(f"!! Google ranges not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    google, _sync = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(google)} Google CIDRs loaded "
      f"({int((google['service'] == 'Google Cloud').sum())} GCP, "
      f"{int((google['service'] == 'GOOGLE').sum())} other Google, syncToken {_sync})")

In [ ]:
# --- cell 06 ---
# Akamai prefixes from cidrs/akamai.csv (IPv4 announced by Akamai's ASNs, via RIPEstat;
# refresh with `python cidrs/fetch_akamai.py`). Schema matches `aws`. service is one of
# AKAMAI (CDN / edge), AKAMAI-PROLEXIC (DDoS scrubbing, can include customer space) or
# AKAMAI-LINODE (rentable cloud VMs -- NOT a trust signal, anyone can host there).
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
_AK_FILE = os.path.join(CIDR_DIR, "akamai.csv")
try:
    akamai = pd.read_csv(_AK_FILE)
    akamai = akamai[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_AK_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! akamai.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    akamai, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(akamai)} Akamai CIDRs loaded" + (f"  (file {_age} days old)" if _age is not None else ""))
print(akamai["service"].value_counts().to_string())

In [ ]:
# --- cell 07 ---
# Cloudflare's official proxy / CDN edge ranges from cidrs/cloudflare.csv (refresh with
# `python cidrs/fetch_cloudflare.py`). Schema matches `aws`; service = CLOUDFLARE.
# Official list only -- 1.1.1.1 DNS, WARP egress and customer BYOIP space are not in it.
# Anyone can front a site with Cloudflare (Workers, Tunnels, R2), so NOT a trust signal.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
_CF_FILE = os.path.join(CIDR_DIR, "cloudflare.csv")
try:
    cloudflare = pd.read_csv(_CF_FILE)
    cloudflare = cloudflare[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_CF_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! cloudflare.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    cloudflare, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(cloudflare)} Cloudflare CIDRs loaded" + (f"  (file {_age} days old)" if _age is not None else ""))
cloudflare

In [ ]:
# --- cell 08 ---
# Port -> service names from conf/tcp-ports.md (protocol 6) and conf/udp-ports.md (17).
# Only lines starting with a port number are read: "port<TAB>service<TAB>description",
# where "67 / 68" lists several ports sharing one name. Headings and column-header
# lines are skipped. A port listed twice in one file keeps its first name (warned).
# Missing / unreadable file -> warn and carry on without that protocol's names.
# Also IP protocol number -> name from conf/protocols.md ("<number> <name>", any spacing;
# everything after the number is the name) -> `protocolNames`, used by cell 14a for
# sessions that aren't TCP / UDP.

CONF_DIR    = "conf"
_PORT_FILES = {6: "tcp-ports.md", 17: "udp-ports.md"}

_rows = []
for _proto, _fn in _PORT_FILES.items():
    try:
        with open(os.path.join(CONF_DIR, _fn), encoding="utf-8") as _f:
            for _no, _ln in enumerate(_f, 1):
                if not _ln.strip()[:1].isdigit():
                    continue
                _parts = [p.strip() for p in re.split(r"\t+|\s{2,}", _ln.strip())]
                try:
                    _ports = [int(p) for p in _parts[0].split("/")]
                    if len(_parts) < 2 or not _parts[1] or not all(0 <= p <= 65535 for p in _ports):
                        raise ValueError
                except ValueError:
                    print(f"!! {_fn} line {_no} not understood, skipped: {_ln.strip()!r}")
                    continue
                _rows += [{"protocol": _proto, "port": p, "service": _parts[1]} for p in _ports]
    except OSError as _e:
        print(f"!! {_fn} not loaded ({type(_e).__name__}: {_e}) -> no protocol-{_proto} service names")

portServices = pd.DataFrame(_rows, columns=["protocol", "port", "service"])
_dupe = portServices.duplicated(["protocol", "port"], keep="first")
for _, _r in portServices[_dupe].iterrows():
    print(f"!! protocol {_r.protocol} port {_r.port} listed again as {_r.service!r} -- keeping the first name")
portServices = portServices[~_dupe].reset_index(drop=True)

protocolNames = {}
try:
    with open(os.path.join(CONF_DIR, "protocols.md"), encoding="utf-8") as _f:
        for _no, _ln in enumerate(_f, 1):
            _m = re.match(r"\s*(\d+)\s+(.+?)\s*$", _ln)
            if not _ln.strip()[:1].isdigit():
                continue
            if not _m or not 0 <= int(_m.group(1)) <= 255:
                print(f"!! protocols.md line {_no} not understood, skipped: {_ln.strip()!r}")
                continue
            if int(_m.group(1)) in protocolNames:
                print(f"!! protocols.md: protocol {_m.group(1)} listed again -- keeping the first name")
                continue
            protocolNames[int(_m.group(1))] = _m.group(2)
except OSError as _e:
    print(f"!! protocols.md not loaded ({type(_e).__name__}: {_e}) -> non-TCP/UDP sessions get no tag-service")

print(f"{len(protocolNames)} protocol names: " + ", ".join(f"{k}={v}" for k, v in sorted(protocolNames.items())))
print(f"{len(portServices)} port names  "
      + ", ".join(f"{_PORT_FILES[k]}: {v}" for k, v in portServices["protocol"].value_counts().sort_index().items()))

In [ ]:
# --- cell 09 ---
# Fastly's official CDN / edge ranges from cidrs/fastly.csv (refresh with
# `python cidrs/fetch_fastly.py`). Schema matches `aws`; service = FASTLY.
# Anyone can serve content or run code (Compute@Edge) via Fastly, so NOT a trust signal.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
_FL_FILE = os.path.join(CIDR_DIR, "fastly.csv")
try:
    fastly = pd.read_csv(_FL_FILE)
    fastly = fastly[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_FL_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! fastly.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    fastly, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(fastly)} Fastly CIDRs loaded" + (f"  (file {_age} days old)" if _age is not None else ""))

In [ ]:
# --- cell 10 ---
# GitHub's own service ranges from cidrs/github.csv (refresh with
# `python cidrs/fetch_github.py`; from api.github.com/meta, Actions / Codespaces left out).
# Schema matches `aws`; service = GITHUB. Several are /32s inside Azure space, which beat
# the broad Azure prefix (most specific wins in cell 12). Repos, gists and raw files make
# GitHub a common exfil / staging channel, so NOT a trust signal.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
_GH_FILE = os.path.join(CIDR_DIR, "github.csv")
try:
    github = pd.read_csv(_GH_FILE)
    github = github[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_GH_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! github.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    github, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(github)} GitHub CIDRs loaded" + (f"  (file {_age} days old)" if _age is not None else ""))

In [ ]:
# --- cell 11 ---
# Oracle Cloud (OCI) ranges from cidrs/oracle.csv (refresh with
# `python cidrs/fetch_oracle.py`; from Oracle's public_ip_ranges.json). Schema matches
# `aws`; service = ORACLE, region = OCI region. Anyone can rent OCI compute or Object
# Storage (free tier included), so NOT a trust signal.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
_OR_FILE = os.path.join(CIDR_DIR, "oracle.csv")
try:
    oracle = pd.read_csv(_OR_FILE)
    oracle = oracle[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_OR_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! oracle.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    oracle, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(oracle)} Oracle CIDRs loaded" + (f"  (file {_age} days old)" if _age is not None else ""))

In [ ]:
# --- cell 12 ---
# ServiceNow ranges from cidrs/servicenow.csv (IPv4 announced by ServiceNow's ASNs, via
# RIPEstat; refresh with `python cidrs/fetch_servicenow.py`). Schema matches `aws`.
# service is SERVICENOW (instance / platform data centres) or SERVICENOW-CORP (the
# company's own network). Free developer instances share the platform space, so NOT a
# trust signal on its own -- confirm the flow is to your own instance.
# Missing / unreadable file -> warn and carry on with an empty frame.

_SCHEMA = ["ip_prefix", "region", "service", "network_border_group"]
_SN_FILE = os.path.join(CIDR_DIR, "servicenow.csv")
try:
    servicenow = pd.read_csv(_SN_FILE)
    servicenow = servicenow[_SCHEMA]
    _age = int((pd.Timestamp.now(tz="UTC") - pd.Timestamp(os.path.getmtime(_SN_FILE), unit="s", tz="UTC")).days)
except (OSError, ValueError, KeyError) as _e:
    print(f"!! servicenow.csv not loaded ({type(_e).__name__}: {_e}) -> empty frame")
    servicenow, _age = pd.DataFrame(columns=_SCHEMA), None

print(f"{len(servicenow)} ServiceNow CIDRs loaded" + (f"  (file {_age} days old)" if _age is not None else ""))
print(servicenow["service"].value_counts().to_string())

In [ ]:
# --- cell 13 ---
# Duplicate rows in vflows, matched on DUP_COLS only. Comment a column out to
# ignore it, e.g. drop interface-id to catch the same flow logged by two ENIs.

DUP_COLS = [
    "version",
    "account-id",
    #"interface-id",
    "srcaddr",
    "dstaddr",
    "srcport",
    "dstport",
    "protocol",
    #"packets",
    #"bytes",
    #"start",
    #"end",
    # "start_time",    # same instant as start
    # "end_time",      # same instant as end
    "action",
    "log-status",
]

_missing = [c for c in DUP_COLS if c not in vflows.columns]
if _missing:
    raise KeyError(f"not in vflows: {_missing}")

_dup   = vflows.duplicated(subset=DUP_COLS, keep=False)
_extra = int(vflows.duplicated(subset=DUP_COLS, keep="first").sum())

print(f"matching on {len(DUP_COLS)} columns: {', '.join(DUP_COLS)}")
print(f"{len(vflows):,} rows")
print(f"{_extra:,} extra copies (rows you'd drop to dedupe)  {_extra / len(vflows):.2%}")
print(f"{int(_dup.sum()):,} rows involved in duplication, "
      f"{int(_dup.sum()) - _extra:,} distinct duplicated records")

vflowsDups = (vflows[_dup].groupby(DUP_COLS, dropna=False).size()
               .rename("copies").sort_values(ascending=False).reset_index())

In [ ]:
# --- cell 15 ---
# Aggregate repeated flow records into sessions -> vflowsAgg (vflows untouched).
# Key = DUP_COLS from the cell above, minus the per-record counters and times.
# Records with the same key merge while each starts within MERGE_GAP seconds of
# the latest end so far; a longer gap starts a new session, so repeat connections
# (scans, beacons, ICMP, reused source ports) keep their timing.
# NODATA (no traffic in the window) and SKIPDATA (records lost at capture) rows carry
# only ENI + time -- set the flags to leave them out of vflowsAgg.
# Fails fast with a pointer to the cell to fix if its inputs aren't what it expects.

MERGE_GAP     = 600     # seconds -- AWS's longest flow-log aggregation window
DROP_NODATA   = False
DROP_SKIPDATA = False

_per_rec = ["packets", "bytes", "start", "end", "start_time", "end_time"]

if "DUP_COLS" not in globals():
    raise NameError("DUP_COLS not defined -- run cell 07 first")
_missing = [c for c in dict.fromkeys(DUP_COLS + _per_rec) if c not in vflows.columns]
if _missing:
    raise KeyError(f"not in vflows: {_missing}")
if vflows.empty:
    raise ValueError("vflows is empty -- nothing to aggregate (check PARQUET_PATH in cell 01)")
_bad = [c for c in ("start", "end", "packets", "bytes")
        if not pd.api.types.is_numeric_dtype(vflows[c]) or pd.api.types.is_bool_dtype(vflows[c])]
if _bad:
    raise TypeError("must be numeric (start/end in epoch seconds): "
                    + ", ".join(f"{c} is {vflows[c].dtype}" for c in _bad) + " -- re-run cell 06")

_v    = vflows[[c for c in vflows.columns if not c.startswith("tag-")]]   # tag-* fields are derived later; log fields (e.g. `type`) are kept
_drop = [s for s, f in (("NODATA", DROP_NODATA), ("SKIPDATA", DROP_SKIPDATA)) if f]
if _drop and "log-status" not in _v.columns:
    print(f"!! no log-status column -- can't drop {', '.join(_drop)}")
elif _drop:
    _m = _v["log-status"].isin(_drop)
    print(f"dropped {int(_m.sum()):,} {'/'.join(_drop)} records (vflows keeps them)")
    _v = _v[~_m]
if _v.empty:
    raise ValueError("no records left to aggregate after dropping " + "/".join(_drop))

_key = [c for c in DUP_COLS if c not in _per_rec]
_noTime = int(_v["start"].isna().sum() + _v["end"].isna().sum())
if _noTime:
    print(f"!! {_noTime:,} missing start/end values -- those records become sessions of their own")

_gid = _v.groupby(_key, dropna=False, sort=False).ngroup()
_v   = _v.assign(_gid=_gid).sort_values(["_gid", "start"], kind="stable")

_prev_end = _v.groupby("_gid")["end"].cummax().groupby(_v["_gid"]).shift()
_new      = (_v["_gid"].ne(_v["_gid"].shift())
             | (_v["start"] - _prev_end).gt(MERGE_GAP).fillna(_v["start"].isna()))
_v["_sid"] = _new.astype(bool).cumsum()

_aggs = {c: (c, "first") for c in _v.columns if c not in _per_rec + ["_gid", "_sid"]}
_aggs.update(start=("start", "min"), end=("end", "max"),
             start_time=("start_time", "min"), end_time=("end_time", "max"),
             records=("start", "size"))

_g   = _v.groupby("_sid", sort=False)
_out = _g.agg(**_aggs)
_out[["packets", "bytes"]] = _g[["packets", "bytes"]].sum(min_count=1)   # NODATA stays <NA>, not 0
# whole-number counts as Int64 (integers that allow <NA>): a plain int column can't hold the
# NODATA <NA>s, so without this they'd be float64 (3704560142.0) here and in every notebook after
_out[["packets", "bytes"]] = _out[["packets", "bytes"]].round().astype("Int64")

vflowsAgg = (_out[[c for c in vflows.columns if c in _out.columns] + ["records"]]
              .sort_values("start", kind="stable")
              .reset_index(drop=True))

if int(vflowsAgg["records"].sum()) != len(_v):
    raise AssertionError(f"record count mismatch: {len(_v):,} in, "
                         f"{int(vflowsAgg['records'].sum()):,} accounted for in vflowsAgg")

_merged = int((vflowsAgg["records"] > 1).sum())
print(f"key ({len(_key)}): {', '.join(_key)}   gap <= {MERGE_GAP}s")
print(f"{len(_v):,} records -> {len(vflowsAgg):,} sessions  "
      f"({len(_v) - len(vflowsAgg):,} folded in, {_merged:,} sessions built from >1 record)")
print(f"bytes   {_v['bytes'].sum():,} -> {vflowsAgg['bytes'].sum():,}")
print(f"packets {_v['packets'].sum():,} -> {vflowsAgg['packets'].sum():,}")

In [ ]:
# --- cell 16 ---
# unique public IPs across ALL flows (either end), from src-int / dst-int (cell 00) --
# no string parsing here. ipv4_is_public() is the one definition of public; the
# direction cell below reuses it. public is sorted by ip_int for the CIDR match downstream.

# IPv4 blocks ipaddress marks non-global (RFC1918, CGNAT, loopback, link-local,
# doc/benchmark ranges, multicast + reserved + broadcast)

_NON_GLOBAL = [
    ("0.0.0.0", "0.255.255.255"),       ("10.0.0.0", "10.255.255.255"),
    ("100.64.0.0", "100.127.255.255"),  ("127.0.0.0", "127.255.255.255"),
    ("169.254.0.0", "169.254.255.255"), ("172.16.0.0", "172.31.255.255"),
    ("192.0.0.0", "192.0.0.255"),       ("192.0.2.0", "192.0.2.255"),
    ("192.88.99.0", "192.88.99.255"),   ("192.168.0.0", "192.168.255.255"),
    ("198.18.0.0", "198.19.255.255"),   ("198.51.100.0", "198.51.100.255"),
    ("203.0.113.0", "203.0.113.255"),   ("224.0.0.0", "255.255.255.255"),
]
_NON_GLOBAL_INT = [(int(ip_address(_lo)), int(ip_address(_hi))) for _lo, _hi in _NON_GLOBAL]

def ipv4_is_public(ints):
    """IPv4 ints (UInt32 column, <NA> allowed) -> bool array; <NA> (IPv6 / missing) -> False."""
    s   = pd.Series(ints, dtype="UInt32")
    v   = s.fillna(0).to_numpy("uint64")
    pub = s.notna().to_numpy(bool)
    for _lo, _hi in _NON_GLOBAL_INT:
        pub &= ~((v >= _lo) & (v <= _hi))
    return pub

if "src-int" not in vflowsAgg.columns or "dst-int" not in vflowsAgg.columns:
    raise KeyError("vflowsAgg has no src-int / dst-int -- re-run cell 00, then from cell 15")

_end = ["ip", "ip_int"]
_v4  = (pd.concat([vflowsAgg[["srcaddr", "src-int"]].set_axis(_end, axis=1),
                   vflowsAgg[["dstaddr", "dst-int"]].set_axis(_end, axis=1)], ignore_index=True)
        .dropna(subset=["ip_int"]).drop_duplicates("ip_int"))

public = (_v4[ipv4_is_public(_v4["ip_int"])]
          .astype({"ip_int": "uint64"})
          .sort_values("ip_int", kind="stable")
          .reset_index(drop=True))
print(f"{len(public)} unique public IPs  (of {len(_v4)} distinct IPv4 across all flows)")
print(public.isna().sum())

In [ ]:
# --- cell 17 ---
# Traffic direction per session in vflowsAgg -> `tag-direction` (see processing.md),
# plus `tag-src-public` / `tag-dst-public` (bool) for each end. Uses ipv4_is_public() from
# the cell above on src-int / dst-int, so public/private has one definition and no strings
# are parsed. Both ends public -> public; exactly one -> north-south; both ends IPv4 and
# non-public -> east-west.
# Address-less / IPv6 rows stay <NA>. (Ingress / egress is the separate `flow-direction`.)

_src_pub = ipv4_is_public(vflowsAgg["src-int"])
_dst_pub = ipv4_is_public(vflowsAgg["dst-int"])
_src_v4  = vflowsAgg["src-int"].notna().to_numpy(bool)
_dst_v4  = vflowsAgg["dst-int"].notna().to_numpy(bool)

vflowsAgg["tag-src-public"] = _src_pub
vflowsAgg["tag-dst-public"] = _dst_pub
vflowsAgg["tag-direction"]  = pd.Series(pd.NA, index=vflowsAgg.index, dtype="string")
vflowsAgg.loc[_src_v4 & _dst_v4 & ~_src_pub & ~_dst_pub, "tag-direction"] = "east-west"
vflowsAgg.loc[_src_pub | _dst_pub, "tag-direction"] = "north-south"
vflowsAgg.loc[_src_pub & _dst_pub, "tag-direction"] = "public"

print(vflowsAgg["tag-direction"].value_counts(dropna=False).to_string())

In [ ]:
# --- cell 18 ---
# Ingress / egress per session in vflowsAgg -> `flow-direction`, the v5 flow log field
# (ENI-relative). If the logs carry it, it's cleaned up in place (lower case, "-" -> <NA>);
# an Athena / Glue `flow_direction` column is renamed to `flow-direction` first.
# v2/v3 logs don't have it, and it is NOT derived -- guessing from which end is public
# isn't reliable -- so those runs have no flow-direction column, and the direction
# filters below are greyed out.

if "flow-direction" not in vflowsAgg.columns and "flow_direction" in vflowsAgg.columns:
    vflowsAgg = vflowsAgg.rename(columns={"flow_direction": "flow-direction"})
    print("flow-direction: renamed from the export's flow_direction column")

if "flow-direction" in vflowsAgg.columns:
    vflowsAgg["flow-direction"] = (vflowsAgg["flow-direction"].astype("string")
                                   .str.strip().str.lower().replace("-", pd.NA))
    print("flow-direction: taken from the log")
    print(vflowsAgg["flow-direction"].value_counts(dropna=False).to_string())
else:
    _v = pd.to_numeric(vflowsAgg["version"], errors="coerce").max()
    print(f"!! no flow-direction field (v{int(_v)} logs) -- not derived; direction filters disabled")

In [ ]:
# --- cell 19 ---
# One interval index over AWS + Zscaler + Tor + Azure + Google + Akamai + Cloudflare + Fastly + GitHub + Oracle + ServiceNow (schema-identical frames).
# Vectorised CIDR -> [start, end]; one binary search per IP; most-specific prefix
# wins, so a Zscaler/Azure /24 beats a broad AWS /12. 
# A reference frame that's undefined, empty or missing columns is reported and left
# out; labelling carries on with the rest (or with none, leaving every tag-org <NA>).
# Writes the matched org / service name to public["tag-org"], and which list matched
# ("aws", "zscaler", "tor", "azure", "google", "akamai", "cloudflare", "fastly",
# "github", "oracle", "servicenow") to public["tag-cloud"]. Cell 20 may override tag-org from known_ips.csv
# (marked by public["known"]) but never touches tag-cloud.

_REF_NAMES = ["aws", "zscaler", "tor", "azure", "google", "akamai", "cloudflare", "fastly", "github", "oracle", "servicenow"]
_REF_NEED  = ["ip_prefix", "service", "region"]

_refOk = []
for _n in _REF_NAMES:
    _df = globals().get(_n)
    if not isinstance(_df, pd.DataFrame):
        print(f"!! {_n} not defined -- run its load cell; no {_n} labels")
    elif _df.empty:
        print(f"!! {_n} is empty -- no {_n} labels")
    elif _miss := [c for c in _REF_NEED if c not in _df.columns]:
        print(f"!! {_n} is missing columns {_miss} -- no {_n} labels")
    else:
        _refOk.append(_n)

if "aws" in _refOk and not (aws["ip_prefix"] == "72.21.192.0/19").any():
    aws = pd.concat([aws, pd.DataFrame([{
        "ip_prefix": "72.21.192.0/19", "region": "-",
        "service": "AMAZON", "network_border_group": "-"}])], ignore_index=True)

refs = (pd.concat([globals()[_n].assign(ref=_n) for _n in _refOk], ignore_index=True)[_REF_NEED + ["ref"]]
        if _refOk else pd.DataFrame(columns=_REF_NEED + ["ref"]))

def cidr_bounds(prefixes, what="prefix"):
    """'a.b.c.d/n' (bare 'a.b.c.d' = /32) -> (start, end, ok): uint64 start / end of each
    range plus a bool mask. The address goes through ipv4_to_uint32 (cell 00), so it is held
    to the same strict IPv4 rule as the flows. IPv6, blank, bad address or a /n outside
    0-32 -> ok False (start = end = 0), reported once with a few examples; callers keep
    only ok rows, so one bad entry can't stop the cell."""
    p    = pd.Series(prefixes, dtype="string").str.strip()
    part = p.str.partition("/")
    lenS = part[2].where(part[1].eq("/"), "32")                     # no "/" -> host /32
    lenS = lenS.where(lenS.str.fullmatch(r"\d{1,2}").fillna(False).astype(bool))
    plen = pd.to_numeric(lenS, errors="coerce")
    ints = ipv4_to_uint32(part[0])
    ok   = (ints.notna() & plen.le(32).fillna(False)).to_numpy(bool)

    base = np.where(ok, ints.fillna(0).to_numpy("uint64"), np.uint64(0))
    plen = np.where(ok, plen.fillna(32).to_numpy(), 32).astype("uint64")
    mask = np.where(
        plen == 0, np.uint64(0),
        (np.uint64(0xFFFFFFFF) << (np.uint64(32) - plen)) & np.uint64(0xFFFFFFFF),
    ).astype("uint64")
    start = base & mask
    end   = np.where(ok, start | (~mask & np.uint64(0xFFFFFFFF)), np.uint64(0))

    if not ok.all():
        bad = p[~ok]
        v6  = int(bad.str.contains(":", regex=False).fillna(False).sum())
        print(f"!! {int((~ok).sum()):,} {what} entries skipped -- not IPv4 CIDR"
              + (f" ({v6:,} IPv6)" if v6 else "") + ", e.g. "
              + ", ".join(repr(x) for x in bad.astype(object).unique()[:3]))
    return start, end, ok

def match_cidr_index(idx, ip_ints):
    """Per IP: row index of the most specific containing prefix, or -1."""
    ip_ints = np.asarray(ip_ints, dtype="uint64")
    starts, ends, cummax = idx["starts"], idx["ends"], idx["cummax"]
    n    = len(starts)
    out  = np.full(len(ip_ints), -1, dtype=np.int64)
    if n == 0:                                                      # no reference data
        return out
    pos  = np.searchsorted(starts, ip_ints, side="right") - 1
    posc = np.clip(pos, 0, n - 1)
    direct  = (pos >= 0) & (ends[posc]   >= ip_ints)
    covered = (pos >= 0) & (cummax[posc] >= ip_ints)
    out[direct] = pos[direct]
    for k in np.where(covered & ~direct)[0]:                        # rare: nested broad block
        j, ipv = int(pos[k]), ip_ints[k]
        while j >= 0 and ends[j] < ipv:
            j -= 1
        out[k] = j
    return out

if len(refs):
    _st, _en, _ok = cidr_bounds(refs["ip_prefix"], "reference")
    if not _ok.all():
        print("   skipped per list: " + ", ".join(
            f"{k} {v}" for k, v in refs.loc[~_ok, "ref"].value_counts().items()))
    refs, _st, _en = refs[_ok].reset_index(drop=True), _st[_ok], _en[_ok]
else:
    _st = _en = np.empty(0, dtype="uint64")

# Sort by start asc, then end desc. For same-start overlaps, searchsorted(side="right")
# lands on the shortest containing prefix, i.e. the most-specific CIDR.

_o = np.lexsort((np.uint64(0xFFFFFFFF) - _en, _st))
ref_idx = {"starts": _st[_o], "ends": _en[_o],
           "cummax": np.maximum.accumulate(_en[_o]) if len(_o) else _en[_o],
           "rows": refs.iloc[_o].reset_index(drop=True)}

public["ref_row"] = match_cidr_index(ref_idx, public["ip_int"].to_numpy("uint64"))
public["in_ref"]  = public["ref_row"] >= 0
_rr  = public["ref_row"].clip(lower=0).to_numpy()
_hit = public["in_ref"].to_numpy()
_rows = ref_idx["rows"] if len(refs) else pd.DataFrame({c: [np.nan] for c in _REF_NEED + ["ref"]})
public["tag-org"]      = np.where(_hit, _rows["service"].to_numpy()[_rr],   np.nan)
public["tag-cloud"]    = np.where(_hit, _rows["ref"].to_numpy()[_rr],       np.nan)
public["ref_prefix"]   = np.where(_hit, _rows["ip_prefix"].to_numpy()[_rr], np.nan)
public["aws_region"]   = np.where(_hit & (public["tag-cloud"].to_numpy() == "aws"),
                                  _rows["region"].to_numpy()[_rr], np.nan)
_lblCols = ["tag-org", "tag-cloud", "ref_prefix", "aws_region"]
public[_lblCols] = public[_lblCols].astype(object)    # stays text-friendly even with no matches

print(f"reference sets used: {', '.join(_refOk) or 'NONE'}  ({len(refs):,} prefixes)")
print(f"matched {int(public['in_ref'].sum())} / {len(public)} public IPs")

In [ ]:
# --- cell 20 ---
# Known / trusted addresses from known_ips.csv (kept out of the notebook so it never gets
# committed). Columns:
#   ip_prefix   IPv4 address (= /32) or CIDR
#   tag-org     owner name; blank -> "known"
#   asn         AS number, digits only ("AS" in front is tolerated); optional
#   cc          2-letter country code; optional. Read as text, so NA stays Namibia
# Rows whose ip_prefix starts with # are skipped.
# Tags `public` with known / known_label / known_asn / known_cc, writes known_label into
# `tag-org` and sets `tag-cloud` to "known" (public["known"] = True marks it), so both flow
# into vflowsAgg. A known entry overrides a cloud list's labels -- it's the more specific
# fact; the CIDR it also matched stays in ref_prefix / in_ref. Where entries overlap, the
# later row wins. Nothing is filtered.
# A bad ip_prefix (IPv6, typo) skips that row; a bad asn / cc is reported and left blank.
# Missing / unreadable file -> warn and carry on with nothing known.

KNOWN_FILE = "known_ips.csv"
_KCOLS     = ["ip_prefix", "tag-org", "asn", "cc"]

try:
    known = pd.read_csv(KNOWN_FILE, dtype=str, keep_default_na=False)
    if _miss := [c for c in _KCOLS if c not in known.columns]:
        raise ValueError(f"missing columns {_miss} -- need {', '.join(_KCOLS)}")
    known = known[_KCOLS].apply(lambda s: s.str.strip())
    known = known[known["ip_prefix"].ne("") & ~known["ip_prefix"].str.startswith("#")].reset_index(drop=True)
except (OSError, ValueError) as _e:
    print(f"!! {KNOWN_FILE} not loaded ({type(_e).__name__}: {_e}) -> nothing known")
    known = pd.DataFrame({c: pd.Series(dtype=object) for c in _KCOLS})

known["tag-org"] = known["tag-org"].mask(known["tag-org"].eq(""), "known")
_asn  = known["asn"].str.replace(r"^AS", "", case=False, regex=True)
_cc   = known["cc"].str.upper()
_badA = _asn.ne("") & ~_asn.str.fullmatch(r"\d{1,10}").fillna(False).astype(bool)
_badC = _cc.ne("") & ~_cc.str.fullmatch(r"[A-Z]{2}").fillna(False).astype(bool)
for _what, _bad, _col in (("asn", _badA, "asn"), ("cc", _badC, "cc")):
    if _bad.any():
        print(f"!! {int(_bad.sum())} bad {_what} values in {KNOWN_FILE} left blank, e.g. "
              + ", ".join(repr(x) for x in known.loc[_bad, _col].unique()[:3]))
known["asn"] = _asn.mask(_badA | _asn.eq(""))
known["cc"]  = _cc.mask(_badC | _cc.eq(""))

public["known"] = False
for _c in ("known_label", "known_asn", "known_cc"):
    public[_c] = pd.Series(pd.NA, index=public.index, dtype="string")
if len(known):
    _ks, _ke, _ok = cidr_bounds(known["ip_prefix"], KNOWN_FILE)
    known, _ks, _ke = known[_ok].reset_index(drop=True), _ks[_ok], _ke[_ok]
    _ip = public["ip_int"].to_numpy("uint64")
    for _s, _e, _org, _ka, _kc in zip(_ks, _ke, known["tag-org"], known["asn"], known["cc"]):
        _hit = (_ip >= _s) & (_ip <= _e)
        public.loc[_hit, "known"]       = True
        public.loc[_hit, "known_label"] = _org
        public.loc[_hit, "known_asn"]   = _ka
        public.loc[_hit, "known_cc"]    = _kc

public.loc[public["known"], "tag-org"]   = public.loc[public["known"], "known_label"]
public.loc[public["known"], "tag-cloud"] = "known"

_kn = public["known"]
print(f"{len(known)} known entries -> {int(_kn.sum())} public IPs tagged "
      f"({int((_kn & public['known_asn'].notna()).sum())} with an asn, "
      f"{int((_kn & public['known_cc'].notna()).sum())} with a cc)")

In [ ]:
# --- cell 20a ---
# Owner and country of every public IP -- cloud-list and known_ips.csv ones too -- from
# iptoasn.com's ip2asn-v4.tsv (public domain, built from BGP
# data; download it into IP2ASN_FILE). One row per address range: start, end, AS number,
# country, AS description. Ranges don't overlap, so one binary search per IP.
# Adds to `public` (cell 21a copies them onto the sessions):
#   tag-asn   AS description, e.g. "CLOUDFLARENET - Cloudflare, Inc."; "unrouted" when no
#             network announces the IP (spoofed / bogon / scanning traffic)
#   tag-cc    country the AS is registered in -- not where the host is, and not the RIR
#             allocation country Cymru reports as cc. "unrouted" when unrouted, "unknown"
#             when ip2asn has no country for a routed range ("Unknown" / "None")
#   tag-country  tag-cc's name from COUNTRY_CODES_FILE (EU / AP, ip2asn's regional codes,
#             are named below); "unrouted" / "unknown" as tag-cc; <NA> if not in the file
# Routing is as of the file's date, not the logs'. IPs not in the file keep <NA>.
# Missing / unreadable file -> warn and
# leave the columns <NA>. Warns when the file is older than IP2ASN_MAX_AGE_DAYS.

IP2ASN_FILE         = os.path.join("cidrs", "ip2asn", "ip2asn-v4.tsv")   # .tsv or .tsv.gz
IP2ASN_MAX_AGE_DAYS = 30
COUNTRY_CODES_FILE  = os.path.join("cidrs", "country_codes.csv")      # cc,country
IP2ASN_COLS         = ["tag-asn", "tag-cc", "tag-country"]

for _c in IP2ASN_COLS:
    public[_c] = pd.Series(pd.NA, index=public.index, dtype="string")

try:
    _ia = pd.read_csv(IP2ASN_FILE, sep="\t", header=None, names=["start", "end", "asn", "cc", "name"],
                      dtype=str, keep_default_na=False, quoting=3)   # quoting=3: names can hold quotes
except (OSError, ValueError) as _e:
    print(f"!! {IP2ASN_FILE} not loaded ({type(_e).__name__}: {_e}) -> no tag-asn / tag-cc")
    _ia = None

if _ia is not None:
    _age = (pd.Timestamp.now() - pd.Timestamp(os.path.getmtime(IP2ASN_FILE), unit="s")).days
    if _age > IP2ASN_MAX_AGE_DAYS:
        print(f"!! {IP2ASN_FILE} is {_age} days old -- download a fresh copy from iptoasn.com")
    _s, _e = ipv4_to_uint32(_ia["start"]), ipv4_to_uint32(_ia["end"])
    _ok = (_s.notna() & _e.notna()).to_numpy(bool)
    if not _ok.all():
        print(f"!! {int((~_ok).sum()):,} unreadable ranges in {IP2ASN_FILE} skipped")
    _ia = (_ia[_ok].assign(s=_s[_ok].to_numpy("uint64"), e=_e[_ok].to_numpy("uint64"))
           .sort_values("s").reset_index(drop=True))

    _ip   = public["ip_int"].to_numpy("uint64")
    _pos  = np.searchsorted(_ia["s"].to_numpy(), _ip, side="right") - 1
    _pc   = np.clip(_pos, 0, None)
    _in   = (_pos >= 0) & (_ip <= _ia["e"].to_numpy()[_pc])
    _rt   = _in & (_ia["asn"].to_numpy()[_pc] != "0")            # asn 0 = "Not routed"
    _cc   = _ia["cc"].to_numpy()[_pc]
    public.loc[_rt, "tag-asn"]        = _ia["name"].to_numpy()[_pc][_rt]
    public.loc[_in & ~_rt, "tag-asn"] = "unrouted"
    _hasCc = _rt & ~np.isin(_cc, ["None", "Unknown"])
    public.loc[_hasCc, "tag-cc"] = _cc[_hasCc]
    public.loc[_rt & ~_hasCc, "tag-cc"] = "unknown"
    public.loc[_in & ~_rt, "tag-cc"]    = "unrouted"

    try:   # keep_default_na=False: "NA" is Namibia
        _ccNames = dict(pd.read_csv(COUNTRY_CODES_FILE, dtype=str, keep_default_na=False)
                        .itertuples(index=False, name=None))
    except (OSError, ValueError) as _e:
        print(f"!! {COUNTRY_CODES_FILE} not loaded ({type(_e).__name__}: {_e}) -> no tag-country")
        _ccNames = {}
    if _ccNames:
        _ccNames = {"EU": "European Union", "AP": "Asia/Pacific Region", **_ccNames}
    _special = {"unrouted": "unrouted", "unknown": "unknown"}
    public["tag-country"] = public["tag-cc"].map({**_ccNames, **_special}).astype("string")
    _unnamed = sorted(set(public.loc[public["tag-cc"].notna() & public["tag-country"].isna(), "tag-cc"]))
    if _ccNames and _unnamed:
        print(f"!! country codes not in {COUNTRY_CODES_FILE} (tag-country left <NA>): {_unnamed}")

    print(f"{len(_ia):,} ranges from {IP2ASN_FILE} ({_age} days old); {len(public):,} public IPs: "
          f"{int(_rt.sum()):,} routed, {int((_in & ~_rt).sum()):,} unrouted, {int((~_in).sum()):,} not in the file")
    _real = public["tag-cc"].notna() & ~public["tag-cc"].isin(list(_special))
    print(f"{public['tag-asn'].nunique():,} distinct tag-asn, {public.loc[_real, 'tag-cc'].nunique():,} countries; "
          f"tag-cc unrouted {int(public['tag-cc'].eq('unrouted').sum()):,}, unknown "
          f"{int(public['tag-cc'].eq('unknown').sum()):,}, {int((_real & public['tag-country'].notna()).sum()):,} / "
          f"{int(_real.sum()):,} country codes named")

In [ ]:
# --- cell 21 ---
# Per-session labels in vflowsAgg, by tag-direction (cell 17) -- see processing.md:
#   north-south  tag-org / tag-cloud from the public end's row in `public` (cell 19's
#                cloud lists, then cell 20's known_ips.csv, which wins and sets tag-cloud
#                "known"). No match -> <NA>. Cell 21a adds ip2asn's tag-asn / tag-cc.
#   public       both ends public -> tag-org = tag-cloud = "public"
#   east-west    tag-org = tag-cloud = "private"; when either end is in reserved space that
#                isn't RFC1918 (100.64/10 CGNAT, documentation / benchmark ranges, ...: the
#                rest of _NON_GLOBAL in cell 16), tag-cloud = "reserved" and tag-org is
#                left <NA> -- tag-org is only set when we know who uses an address.
#   no address   (NODATA / SKIPDATA, IPv6; tag-direction <NA>) -> <NA>
# A north-south session has exactly one public end: srcaddr if it's in `public`, else dstaddr.

_pubIp = public.set_index("ip")
_end   = vflowsAgg["srcaddr"].where(vflowsAgg["srcaddr"].isin(_pubIp.index),
                                    vflowsAgg["dstaddr"].where(vflowsAgg["dstaddr"].isin(_pubIp.index)))
_dir   = vflowsAgg["tag-direction"]
_ns    = _dir.eq("north-south").fillna(False)

vflowsAgg["tag-org"]    = _end.where(_ns).map(_pubIp["tag-org"]).astype("string")
vflowsAgg["tag-cloud"]  = _end.where(_ns).map(_pubIp["tag-cloud"]).astype("string")
_known = vflowsAgg["tag-cloud"].eq("known").fillna(False)
vflowsAgg = vflowsAgg.drop(columns=["tag-source"], errors="ignore")   # older runs of this cell made it
for _d, _v in (("public", "public"), ("east-west", "private")):
    vflowsAgg.loc[_dir.eq(_d).fillna(False), ["tag-org", "tag-cloud"]] = _v

_RFC1918_INT = [(int(ip_address(_lo)), int(ip_address(_hi))) for _lo, _hi in
                (("10.0.0.0", "10.255.255.255"), ("172.16.0.0", "172.31.255.255"),
                 ("192.168.0.0", "192.168.255.255"))]

def ipv4_is_reserved(ints):
    """IPv4 ints -> bool array: not public and not RFC1918; <NA> (IPv6 / missing) -> False."""
    s   = pd.Series(ints, dtype="UInt32")
    v   = s.fillna(0).to_numpy("uint64")
    res = s.notna().to_numpy(bool) & ~ipv4_is_public(s)
    for _lo, _hi in _RFC1918_INT:
        res &= ~((v >= _lo) & (v <= _hi))
    return res

_res = (_dir.eq("east-west").fillna(False).to_numpy(bool)
        & (ipv4_is_reserved(vflowsAgg["src-int"]) | ipv4_is_reserved(vflowsAgg["dst-int"])))
vflowsAgg.loc[_res, "tag-cloud"] = "reserved"
vflowsAgg.loc[_res, "tag-org"]   = pd.NA

print(f"{int(_ns.sum()):,} north-south sessions: {int((_ns & vflowsAgg['tag-cloud'].notna() & ~_known).sum()):,} "
      f"on a cloud list, {int(_known.sum()):,} known_ips.csv, {int((_ns & vflowsAgg['tag-cloud'].isna()).sum()):,} "
      f"on neither")
print(f"{int(_res.sum()):,} east-west sessions with an end in reserved (non-RFC1918) space -> "
      f"tag-cloud reserved")
print(vflowsAgg["tag-cloud"].value_counts(dropna=False).to_string())

In [ ]:
# --- cell 21a ---
# ip2asn labels (cell 20a) on every north-south session: tag-asn, tag-cc and tag-country from the public
# end's row in `public`. Every other session (public, east-west, no address) keeps <NA>.

if "tag-asn" not in public.columns:
    raise KeyError("public has no tag-asn / tag-cc -- run cell 20a first")

_pubIp = public.set_index("ip")
_end   = vflowsAgg["srcaddr"].where(vflowsAgg["srcaddr"].isin(_pubIp.index),
                                    vflowsAgg["dstaddr"].where(vflowsAgg["dstaddr"].isin(_pubIp.index)))
_m     = vflowsAgg["tag-direction"].eq("north-south").fillna(False)
for _c in IP2ASN_COLS:
    vflowsAgg[_c] = _end.where(_m).map(_pubIp[_c]).astype("string")

print(f"{int(_m.sum()):,} north-south sessions: {int(vflowsAgg['tag-asn'].notna().sum()):,} "
      f"got a tag-asn ({int(vflowsAgg['tag-asn'].eq('unrouted').sum()):,} unrouted), "
      f"{int(vflowsAgg['tag-cc'].notna().sum()):,} a tag-cc, {int(vflowsAgg['tag-country'].notna().sum()):,} a tag-country")
print(vflowsAgg.loc[_m].groupby(["tag-cc", "tag-country"], dropna=False).size()
      .sort_values(ascending=False).head(10).to_string())

In [ ]:
# --- cell 21b ---
# `conflict` = True where a cloud list (cell 19) and ip2asn (cell 21a) disagree about who
# owns the public end of a north-south session, so those can be reviewed later:
#   - the tag-asn name contains none of the keywords for the session's tag-cloud, or
#   - tag-asn is "unrouted" (a cloud list claims an address nobody announces today).
# Keywords are case-insensitive substrings of the ip2asn AS name; edit as needed. Clouds not
# listed here (tor, known, ...) have no expected owner and are never flagged -- Tor relays
# run on anyone's network. Everything else, including <NA> tag-asn, is False.
# Expect some legitimate hits: Akamai edge caches inside ISPs (Reliance Jio, ...), AWS
# China regions run by Sinnet / NWCD, Zscaler nodes in Google Cloud.
CONFLICT_KEYWORDS = {
    "aws":        ["AMAZON"],
    "akamai":     ["AKAMAI", "PROLEXIC"],
    "azure":      ["MICROSOFT"],
    "google":     ["GOOGLE", "YOUTUBE"],
    "cloudflare": ["CLOUDFLARE"],
    "fastly":     ["FASTLY"],
    "github":     ["GITHUB"],
    "oracle":     ["ORACLE"],
    "servicenow": ["SERVICENOW"],
    "zscaler":    ["ZSCALER"],
}

if "tag-asn" not in vflowsAgg.columns:
    raise KeyError("vflowsAgg has no tag-asn -- run cell 21a first")

_cloud = vflowsAgg["tag-cloud"]
_asnUp = vflowsAgg["tag-asn"].str.upper()
vflowsAgg["conflict"] = False
for _cl, _kw in CONFLICT_KEYWORDS.items():
    _r = _cloud.eq(_cl).fillna(False) & vflowsAgg["tag-asn"].notna()
    if not _r.any():
        continue
    _ok = pd.Series(False, index=vflowsAgg.index)
    for _k in _kw:
        _ok |= _asnUp.str.contains(_k.upper(), regex=False).fillna(False)
    vflowsAgg.loc[_r & (~_ok | vflowsAgg["tag-asn"].eq("unrouted").fillna(False)), "conflict"] = True

_unchecked = sorted(set(_cloud.dropna().unique()) - set(CONFLICT_KEYWORDS) - {"public", "private", "reserved"})
_cf  = vflowsAgg["conflict"]
_cfIp = vflowsAgg.loc[_cf, "srcaddr"].where(vflowsAgg.loc[_cf, "srcaddr"].isin(public["ip"]),
                                            vflowsAgg.loc[_cf, "dstaddr"])
print(f"{int(_cf.sum()):,} sessions with conflict = True, {_cfIp.nunique():,} distinct public IPs "
      f"({int((_cf & vflowsAgg['tag-asn'].eq('unrouted').fillna(False)).sum()):,} sessions unrouted); "
      f"not checked: {', '.join(_unchecked) or 'none'}")
if _cf.any():
    print(vflowsAgg.loc[_cf].groupby(["tag-cloud", "tag-asn"]).size()
          .sort_values(ascending=False).head(15).to_string())

In [ ]:
# --- cell 22 ---
# Service name per session in vflowsAgg -> `tag-service`, from portServices (cell 05c),
# matched on (protocol, port) at both ends:
#   one end's port listed            -> that service
#   both listed, same service        -> that service (e.g. NTP 123 <-> 123)
#   both listed, DIFFERENT services  -> "anomalous" (scanners sending from a fixed
#                                       well-known port, e.g. 443 -> 22)
#   neither listed                   -> <NA>
# Sessions that are neither TCP (6) nor UDP (17) -- ICMP, GRE, tunnels, ... -- get the
# protocol's name from protocolNames (conf/protocols.md) instead; <NA> if it isn't listed.

# (protocol, port) -> one int key, protocol * 65536 + port. A port outside 0-65535 or a
# protocol outside 0-255 can't be a real one (and would collide with another key), so -> <NA>.
def _svcKey(proto, port):
    proto, port = proto.astype("Int64"), port.astype("Int64")
    return (proto * 65536 + port).where(proto.between(0, 255) & port.between(0, 65535))

_svc    = portServices.set_index(_svcKey(portServices["protocol"], portServices["port"]))["service"]
_byPort = lambda col: _svcKey(vflowsAgg["protocol"], vflowsAgg[col]).map(_svc).astype("string")
_srcSvc, _dstSvc = _byPort("srcport"), _byPort("dstport")

vflowsAgg["tag-service"] = _dstSvc.fillna(_srcSvc)
vflowsAgg.loc[(_srcSvc.notna() & _dstSvc.notna() & _srcSvc.ne(_dstSvc)).fillna(False), "tag-service"] = "anomalous"

_otherProto = vflowsAgg["protocol"].notna() & ~vflowsAgg["protocol"].isin([6, 17])
vflowsAgg.loc[_otherProto, "tag-service"] = vflowsAgg.loc[_otherProto, "protocol"].map(protocolNames).astype("string")
_unnamed = sorted(vflowsAgg.loc[_otherProto & vflowsAgg["tag-service"].isna(), "protocol"].unique().tolist())
if _unnamed:
    print(f"!! protocols not in conf/protocols.md (left <NA>): {_unnamed}")

_anom = vflowsAgg["tag-service"].eq("anomalous").fillna(False)
print(f"{int(vflowsAgg['tag-service'].notna().sum()):,} of {len(vflowsAgg):,} sessions named; "
      f"{int(_anom.sum()):,} anomalous ({int((_anom & vflowsAgg['action'].eq('ACCEPT').fillna(False)).sum()):,} ACCEPTed)")
print(vflowsAgg["tag-service"].value_counts(dropna=False).head(5).to_string())

In [ ]:
# --- cell 23 ---
# Distinct reference prefixes that matched at least one public IP

_hit_rows   = np.unique(public.loc[public["in_ref"], "ref_row"].to_numpy())
refMatches = ref_idx["rows"].iloc[_hit_rows].reset_index(drop=True)
print(refMatches["ref"].value_counts().to_string())

In [ ]:
# --- cell 24 ---
# tag-org / tag-cloud / aws_region are set by the combined-match cell above.
# (Precedence is most-specific-prefix-wins; AWS/Azure/Zscaler address space is
#  effectively disjoint, so tag-cloud is unambiguous in practice.)

print(public["tag-cloud"].value_counts(dropna=False).to_string())
print(f'{int(public["known"].sum())} IPs named from known_ips.csv')
print()
print(public["tag-org"].value_counts(dropna=False).head(10).to_string())

In [ ]:
# --- cell 25 ---
# Sessions and bytes by tag-cloud (vflowsAgg) -- a static summary; the interactive
# version lives with the analytics now. Grouped by which reference list the public end
# matched: aws, azure, google, akamai, cloudflare, fastly, github, oracle, servicenow,
# zscaler, tor. Two panels sharing the cloud axis -- never one chart with two scales.
# Fixed view: north-south sessions only (east-west has no public end, so no tag-cloud),
# action CLD_ACTION, linear x, top CLD_TOP_N clouds by sessions. `cloudSummary` holds
# every cloud regardless of CLD_TOP_N.
# Colors follow the cloud, not its rank: the 5 busiest get their own color, others are
# light gray, mid gray = no cloud (public end isn't in any list -- known_ips.csv names
# alone don't count). Every bar is also labeled, so color is never the only cue.
# matplotlib PNG -- no widgets or plotly JS, so it's cheap to render and store.

if "tag-cloud" not in vflowsAgg.columns:
    raise KeyError("vflowsAgg has no 'tag-cloud' column -- re-run cells 12-14 (or everything from cell 08) first")
import matplotlib.pyplot as plt
from matplotlib.ticker import EngFormatter

CLD_ACTION = "ACCEPT"     # "ACCEPT", "REJECT" or None for both
CLD_TOP_N  = 10           # 0 = every cloud

# validated (dataviz validator, all pairs): blue, yellow, magenta, violet, aqua.
_CLD_PALETTE = ["#2a78d6", "#eda100", "#e87ba4", "#4a3aa7", "#1baf7a"]
_CLD_OTHER   = "#c3c2b7"                                                  # remaining clouds
_CLD_COLORS  = {"(no cloud)": "#898781"}                                    # muted gray
_cldRank = vflowsAgg["tag-cloud"].dropna().value_counts().index[:len(_CLD_PALETTE)]
_CLD_COLORS.update(dict(zip(_cldRank, _CLD_PALETTE)))
_INK, _INK2, _GRID, _SURF = "#0b0b0b", "#52514e", "#e8e7e2", "#fcfcfb"


def _fmtBytes(b):
    for u in ("B", "kB", "MB", "GB", "TB"):
        if abs(b) < 1000 or u == "TB":
            return f"{b:,.0f} {u}" if u == "B" else f"{b:,.1f} {u}"
        b /= 1000


_sel = vflowsAgg[vflowsAgg["tag-direction"].eq("north-south").fillna(False)]
if CLD_ACTION:
    _sel = _sel[_sel["action"].eq(CLD_ACTION).fillna(False)]

cloudSummary = (_sel.assign(cloud=_sel["tag-cloud"].astype("string").fillna("(no cloud)"))
                .groupby("cloud")
                .agg(sessions=("records", "size"), bytes=("bytes", "sum"), records=("records", "sum"))
                .sort_values(["sessions", "bytes"], ascending=False))
cloudSummary["bytes"] = cloudSummary["bytes"].fillna(0).astype("int64")
cloudSummary["sessionShare"] = cloudSummary["sessions"] / max(int(cloudSummary["sessions"].sum()), 1)
cloudSummary["byteShare"]    = cloudSummary["bytes"] / max(int(cloudSummary["bytes"].sum()), 1)

_top = cloudSummary.head(CLD_TOP_N) if CLD_TOP_N else cloudSummary
if _top.empty:
    print(f"no north-south{' ' + CLD_ACTION if CLD_ACTION else ''} sessions -- nothing to plot")
else:
    _names  = _top.index.tolist()[::-1]                                    # busiest on top
    _colors = [_CLD_COLORS.get(n, _CLD_OTHER) for n in _names]
    _fig, _axes = plt.subplots(1, 2, sharey=True, figsize=(11, 0.32 * len(_names) + 1.3),
                               facecolor=_SURF, layout="constrained")
    for _ax, _col, _lbl, _fmt, _unit in (
            (_axes[0], "sessions", "Sessions", lambda v: f"{v:,}", ""),
            (_axes[1], "bytes",    "Bytes",    _fmtBytes,         "B")):
        _v = _top[_col].to_numpy()[::-1]
        _ax.barh(_names, _v, color=_colors, height=0.65)
        for _y, _x in enumerate(_v):
            _ax.text(_x, _y, "  " + _fmt(int(_x)), va="center", fontsize=9, color=_INK2)
        _ax.set_facecolor(_SURF)
        _ax.set_title(_lbl, fontsize=11, color=_INK, loc="left")
        _ax.set_xlim(0, max(_v.max(), 1) * 1.25)                           # room for the labels
        _ax.xaxis.set_major_formatter(EngFormatter(unit=_unit, sep=""))     # 1 GB, not 1e9
        _ax.grid(axis="x", color=_GRID, linewidth=1)
        _ax.set_axisbelow(True)
        _ax.tick_params(colors=_INK2, labelsize=9, length=0)
        for _s in _ax.spines.values():
            _s.set_visible(False)
    _axes[0].tick_params(axis="y", colors=_INK, labelsize=10)
    _shown = (f"top {len(_top)} of {len(cloudSummary)} clouds" if len(_top) < len(cloudSummary)
              else f"all {len(cloudSummary)} clouds")
    _fig.suptitle(f"Sessions and bytes by tag-cloud  ({_shown}, {len(_sel):,} north-south "
                  f"{CLD_ACTION or ''} sessions, {_fmtBytes(int(_sel['bytes'].sum()))})",
                  x=0.01, ha="left", fontsize=12, color=_INK)
    plt.show()

In [ ]:
# --- cell 26 ---
# Save vflowsAgg for the analysis notebook (flows-hunting-mark-two) to
# vflowsAgg-<account-id>-<month>-<year>.parquet in OUT_DIR, e.g.
# vflowsAgg-637602092272-october-2026.parquet. Month / year are when this cell runs.
# Several accounts in the data -> their ids joined with "+". Re-running in the same month
# for the same account overwrites that month's file.
# Parquet keeps dtypes (Int64 epochs, UTC timestamps), which CSV would lose.
# OUT_PATH is kept for cell 27, which writes its subset next to it.
# Also writes `public` (one row per public IP with its CIDR / known_ips.csv labels) next
# to it as <same name>-ips.parquet, so later notebooks load the per-IP labels instead of
# rebuilding them from sessions. Includes tag-asn / tag-cc / tag-country when cell 20a ran.
# Splits that file in two by what each IP's sessions did (action, from vflowsAgg; an IP's
# public end is srcaddr when tag-src-public, dstaddr when tag-dst-public):
#   <same name>-ips-accept.parquet       IPs with at least one ACCEPT session
#   <same name>-ips-reject-only.parquet  IPs whose sessions were all REJECT
# Same columns as -ips.parquet. The full file stays -- flows-nbad-cymru cell 03 reads it.

OUT_DIR = "."

if "account-id" not in vflowsAgg.columns:
    raise KeyError("vflowsAgg has no 'account-id' column -- keep it in DUP_COLS (cell 07) and re-run from cell 15")
_accts = sorted(vflowsAgg["account-id"].dropna().astype(str).str.strip().unique())
_accts = [a for a in _accts if a and a != "-"]          # NODATA / SKIPDATA rows can carry "-"
if not _accts:
    raise ValueError("no account-id values in vflowsAgg -- can't name the file")
_now  = pd.Timestamp.now()
_name = f"vflowsAgg-{'+'.join(_accts)}-{_now.strftime('%B').lower()}-{_now.year}.parquet"
OUT_PATH = os.path.join(OUT_DIR, _name)

vflowsAgg.to_parquet(OUT_PATH, index=False)
print(f"wrote {len(vflowsAgg):,} sessions x {vflowsAgg.shape[1]} columns -> {OUT_PATH}  "
      f"({os.path.getsize(OUT_PATH) / 1e6:,.1f} MB)")

_IP_COLS = ["ip", "ip_int", "tag-org", "tag-cloud", "in_ref", "ref_prefix", "aws_region",
            "known", "known_label", "known_asn", "known_cc"]
if _miss := [c for c in _IP_COLS if c not in public.columns]:
    raise KeyError(f"public is missing {_miss} -- re-run cells 16-20")
_IP_COLS += [c for c in globals().get("IP2ASN_COLS", []) if c in public.columns]   # cell 20a
OUT_IPS_PATH = os.path.splitext(OUT_PATH)[0] + "-ips.parquet"
public[_IP_COLS].to_parquet(OUT_IPS_PATH, index=False)
print(f"wrote {len(public):,} public IPs x {len(_IP_COLS)} columns -> {OUT_IPS_PATH}  "
      f"({os.path.getsize(OUT_IPS_PATH) / 1e6:,.1f} MB)")

# which public IPs had an ACCEPT / a REJECT session
_srcPub = vflowsAgg["tag-src-public"].fillna(False).to_numpy(bool)
_dstPub = vflowsAgg["tag-dst-public"].fillna(False).to_numpy(bool)
_pubInt = pd.Series(np.where(_srcPub, vflowsAgg["src-int"].astype("Int64").fillna(-1).to_numpy("int64"),
                             np.where(_dstPub, vflowsAgg["dst-int"].astype("Int64").fillna(-1).to_numpy("int64"), -1)))
_act    = vflowsAgg["action"].reset_index(drop=True)
_accInt = np.unique(_pubInt[_act.eq("ACCEPT").fillna(False) & _pubInt.ge(0)].to_numpy("int64"))
_rejInt = np.unique(_pubInt[_act.eq("REJECT").fillna(False) & _pubInt.ge(0)].to_numpy("int64"))

_ipInt   = public["ip_int"].to_numpy("int64")
_hasAcc  = np.isin(_ipInt, _accInt)
_rejOnly = ~_hasAcc & np.isin(_ipInt, _rejInt)
if (_neither := int((~_hasAcc & ~_rejOnly).sum())):
    print(f"!! {_neither:,} public IPs had neither an ACCEPT nor a REJECT session -- in neither split file")

for _suffix, _rows, _what in (("-ips-accept", _hasAcc, "with at least one ACCEPT"),
                              ("-ips-reject-only", _rejOnly, "REJECT only")):
    _path = os.path.splitext(OUT_PATH)[0] + _suffix + ".parquet"
    public.loc[_rows, _IP_COLS].to_parquet(_path, index=False)
    print(f"wrote {int(_rows.sum()):,} public IPs ({_what}) -> {_path}  "
          f"({os.path.getsize(_path) / 1e6:,.1f} MB)")

In [ ]:
# --- cell 27 ---
# Sessions with no tag-cloud, action NC_ACTION and tag-direction NC_DIRECTION ->
# vflowsNoCloud (a filtered copy; vflowsAgg is untouched). Set either to None to not
# filter on it.
# tag-cloud is <NA> when the public end matched no reference list, and also for east-west
# sessions (no public end) and address-less ones -- the direction breakdown shows the mix.
# Counts the distinct IPs at either end, public vs private, with ipv4_is_public (cell 16),
# the same definition the direction cell uses. Non-IPv4 / missing addresses are counted apart.
# Also counts distinct ports: srcport, dstport, and either end (<NA> ports left out).
# tag-org counts show which sessions carry a known_ips.csv label (cell 20).
# Writes vflowsNoCloud next to cell 26's file, same name + "-subset".

NC_ACTION    = "ACCEPT"         # "ACCEPT", "REJECT" or None for both
NC_DIRECTION = "north-south"    # "north-south", "east-west" or None for both

if "tag-cloud" not in vflowsAgg.columns:
    raise KeyError("vflowsAgg has no 'tag-cloud' column -- run cell 21 first")

_m = vflowsAgg["tag-cloud"].isna()
if NC_ACTION:
    _m &= vflowsAgg["action"].eq(NC_ACTION).fillna(False)
if NC_DIRECTION:
    _m &= vflowsAgg["tag-direction"].eq(NC_DIRECTION).fillna(False)
vflowsNoCloud = vflowsAgg[_m].copy()

_ends = pd.concat([vflowsNoCloud["src-int"], vflowsNoCloud["dst-int"]], ignore_index=True)
_ints = _ends.dropna().unique()
_pub  = ipv4_is_public(_ints)

print(f"{len(vflowsNoCloud):,} of {len(vflowsAgg):,} sessions have no tag-cloud"
      + (f", action {NC_ACTION}" if NC_ACTION else "")
      + (f", direction {NC_DIRECTION}" if NC_DIRECTION else ""))
print(f"distinct IPs: {int(_pub.sum()):,} public, {int((~_pub).sum()):,} private")
if _ends.isna().any():
    print(f"   + {int(_ends.isna().sum()):,} session ends with no IPv4 address (IPv6 / missing)")
print(f"distinct ports: {vflowsNoCloud['srcport'].nunique():,} src, "
      f"{vflowsNoCloud['dstport'].nunique():,} dst, "
      f"{pd.concat([vflowsNoCloud['srcport'], vflowsNoCloud['dstport']]).nunique():,} at either end")
print(vflowsNoCloud["tag-org"].value_counts(dropna=False).to_string())

if "OUT_PATH" not in globals():
    raise NameError("OUT_PATH not set -- run cell 26 first (the subset file is named after its file)")
_root, _ext = os.path.splitext(OUT_PATH)
_subPath    = f"{_root}-subset{_ext}"
vflowsNoCloud.to_parquet(_subPath, index=False)
print(f"wrote {len(vflowsNoCloud):,} sessions x {vflowsNoCloud.shape[1]} columns -> {_subPath}  "
      f"({os.path.getsize(_subPath) / 1e6:,.1f} MB)")